In [32]:
import pandas as pd
import numpy as np
import re

# Property Data

In [33]:
df_raw = pd.read_csv("2024-08-Buy-property_info.csv", encoding="utf-8-sig")
df_raw.drop(columns=["Unnamed: 0.1", "Unnamed: 0"], inplace=True)
print(len(df_raw))
df_raw.head()

89652


,Location,URL,SKU,Title,Price,Category,Num_Bedrooms,Num_Bathrooms,Floor_Area,Land_Area,Geo_Locations,Description,Agent_Name,Agent_Links,Agent_Verifications,File
0,"Amparo, Caloocan",https://www.lamudi.com.ph/projects/amparo-resi...,HO5CEE34F0060ECPH,7.2M single attached house and lot for sale at...,7200000,house,3.0,2.0,147.0,103.0,"[121.0769537,14.7471848]","Project- ELIORA , AMPARO SUBDIVISION Location-...",Evelyn Samaniego,https://www.lamudi.com.ph/evelyn-samaniego/,FULLY VERIFIED,Caloocan
1,Caloocan,https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66D1B50AECE7EPH,Caloocan City RFO Condo beside Lrt Monumento,3100000,condo,1.0,1.0,21.7,NaN,"[120.99083,14.65739]",Reasons why you should buy condo here at Torre...,Mariz Lapuz,https://www.lamudi.com.ph/-agn-36444/,FULLY VERIFIED,Caloocan
2,"Amparo, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,LA66C596CFA8463PH,"850sqm Vacant Lot in Makabud Street, Amparo, N...",17000000,land,NaN,NaN,NaN,850.0,"[121.0769537,14.7471848]",Lot Area: 850 sqm Frontage: 24.54 m Located in...,Paolo Bellosillo,https://www.lamudi.com.ph/-25377/,FULLY VERIFIED,Caloocan
3,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C1D00AA6DD1PH,Pre Selling 2BR 97sqm in Caloocan City near Ar...,11900000,condo,3.0,2.0,97.0,NaN,"[120.9905434,14.6514006]",CALINEA TOWER PRE SELLING 3 BEDROOM 97 SQM IN ...,Ailyn Vigilla,https://www.lamudi.com.ph/ailyn-vigilla-agn-15...,FULLY VERIFIED,Caloocan
4,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C6B155BF32BPH,3BR 97sqm Pre Selling in Caloocan City near Sq...,11900000,condo,3.0,2.0,97.0,NaN,"[120.9905434,14.6514006]",CALINEA TOWER PRE SELLING 3 BEDROOM 97 SQM IN ...,Ailyn Vigilla,https://www.lamudi.com.ph/ailyn-vigilla-agn-15...,FULLY VERIFIED,Caloocan


In [34]:
df = df_raw.copy()

## Data Cleaning

In [35]:
for column in df.columns:
    print(column, df.columns.isna().sum())

Location 0
URL 0
SKU 0
Title 0
Price 0
Category 0
Num_Bedrooms 0
Num_Bathrooms 0
Floor_Area 0
Land_Area 0
Geo_Locations 0
Description 0
Agent_Name 0
Agent_Links 0
Agent_Verifications 0
File 0


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89652 entries, 0 to 89651
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Location             89652 non-null  object 
 1   URL                  89652 non-null  object 
 2   SKU                  89652 non-null  object 
 3   Title                89647 non-null  object 
 4   Price                89652 non-null  object 
 5   Category             89652 non-null  object 
 6   Num_Bedrooms         75491 non-null  float64
 7   Num_Bathrooms        71399 non-null  float64
 8   Floor_Area           78685 non-null  float64
 9   Land_Area            58203 non-null  float64
 10  Geo_Locations        89652 non-null  object 
 11  Description          89469 non-null  object 
 12  Agent_Name           89652 non-null  object 
 13  Agent_Links          89652 non-null  object 
 14  Agent_Verifications  89652 non-null  object 
 15  File                 89652 non-null 

### Removal

In [37]:
print(len(df))
df = df[-(df["Price"]=="Contact agent for price")]
print(len(df))

89652
87136


In [38]:
land_check = df[(df["Category"]=="Land") & (df["Land_Area"].isna() | df["Land_Area"]==0)]
print(len(land_check))

0


### Fix Data Type & Fill Null

In [39]:
for var in ["Price", "Num_Bedrooms", "Num_Bathrooms", "Floor_Area", "Land_Area"]:
    df[var].fillna(0, inplace=True)
    df[var] = pd.to_numeric(df[var], errors='coerce')
for var in ["Title", "Description"]:
    df[var].fillna("", inplace=True)
    df[var] = df[var].astype(str)

In [40]:
def split_geo_locations(location):
    location = location.strip("[]").split(",")
    return float(location[0]), float(location[1])
df[['Latitude', 'Longitude']] = df['Geo_Locations'].apply(lambda x: pd.Series(split_geo_locations(x)))
df.drop(columns="Geo_Locations", inplace=True)

In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 87136 entries, 0 to 89651
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Location             87136 non-null  object 
 1   URL                  87136 non-null  object 
 2   SKU                  87136 non-null  object 
 3   Title                87136 non-null  object 
 4   Price                87136 non-null  float64
 5   Category             87136 non-null  object 
 6   Num_Bedrooms         87136 non-null  float64
 7   Num_Bathrooms        87136 non-null  float64
 8   Floor_Area           87136 non-null  float64
 9   Land_Area            87136 non-null  float64
 10  Description          87136 non-null  object 
 11  Agent_Name           87136 non-null  object 
 12  Agent_Links          87136 non-null  object 
 13  Agent_Verifications  87136 non-null  object 
 14  File                 87136 non-null  object 
 15  Latitude             87136 non-null 

## Duplicate Check

In [42]:
df = df.drop_duplicates()
print(len(df))

86877


### SKU

In [43]:
SKU_duplicates = df["SKU"].duplicated()
print(len(df[SKU_duplicates]))
df[SKU_duplicates].sort_values("SKU").head()

141


,Location,URL,SKU,Title,Price,Category,Num_Bedrooms,Num_Bathrooms,Floor_Area,Land_Area,Description,Agent_Name,Agent_Links,Agent_Verifications,File,Latitude,Longitude
72700,"Rosario, Pasig",https://www.lamudi.com.ph/buy/metro-manila/pas...,AP5C4B0B9D50BCEPH,Dormitory For Sale at the Heart of Pasig.,27000000.0,apartment,16.0,7.0,350.0,0.0,DORMITORY for Sale at the Heart of Pasig.Lifeh...,Ludivica A. Serrano,https://www.lamudi.com.ph/ludivica-a-serrano/,FULLY VERIFIED,Rizal,121.089097,14.588464
30518,"Valenzuela, Makati",https://www.lamudi.com.ph/buy/metro-manila/mak...,AP62418DAA64179PH,For Sale 6 Door Apartments with Tenants in Bar...,60000000.0,apartment,6.0,6.0,320.0,0.0,320 SQM Taguig St. Barangay Valenzuela Village...,Christine Baliza,https://www.lamudi.com.ph/christine-baliza-agn/,FULLY VERIFIED,Mandaluyong,121.024192,14.571361
79014,"Santo Niño, Cainta",https://www.lamudi.com.ph/buy/rizal/cainta/for...,AP6243F5E1D37CFPH,For Sale Apartment in Cainta Rizal,27000000.0,apartment,24.0,0.0,515.0,0.0,4-storey apartment Floor area per unit 38-40 ...,Rose Thomas,https://www.lamudi.com.ph/my-hometown-realty-d...,SEMI VERIFIED,San Juan,121.119199,14.587325
88271,"Pembo, Makati",https://www.lamudi.com.ph/buy/metro-manila/mak...,AP62EB8959B7CCAPH,"San Francisco Street, Pembo Dormitory Building...",28000000.0,apartment,10.0,19.0,300.0,0.0,"San Francisco Street, Pembo, Makati City Lot a...",Jose Mari Gutierrez,https://www.lamudi.com.ph/-agn-49566/,FULLY VERIFIED,Taguig,121.057819,14.544140
30516,"Valenzuela, Makati",https://www.lamudi.com.ph/buy/metro-manila/mak...,AP631FF4F3D0F39PH,Apartment For Sale in Makati,45000000.0,apartment,4.0,5.0,343.0,0.0,"APARTMENT FOR SALE PHP45,000.000,00 NET 343 SQ...",Gloria Balane-Rafer,https://www.lamudi.com.ph/gloria-balane-rafer-...,FULLY VERIFIED,Mandaluyong,121.024192,14.571361


In [44]:
print(len(df))
df = df[~SKU_duplicates]
print(len(df))

86877
86736


In [45]:
# df.to_csv("20241118 Processed File v1 - No SKU Duplicates.csv", encoding="utf-8-sig", index=False)

### Description & Title & Others

In [46]:
df["Concat"] = df.iloc[:, 4:10].apply(lambda row: ' '.join(row.values.astype(str)), axis=1)
concat_duplicates = df["Concat"].duplicated()
df[concat_duplicates].head()

,Location,URL,SKU,Title,Price,Category,Num_Bedrooms,Num_Bathrooms,Floor_Area,Land_Area,Description,Agent_Name,Agent_Links,Agent_Verifications,File,Latitude,Longitude,Concat
4,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C6B155BF32BPH,3BR 97sqm Pre Selling in Caloocan City near Sq...,11900000.0,condo,3.0,2.0,97.00,0.00,CALINEA TOWER PRE SELLING 3 BEDROOM 97 SQM IN ...,Ailyn Vigilla,https://www.lamudi.com.ph/ailyn-vigilla-agn-15...,FULLY VERIFIED,Caloocan,120.990543,14.651401,11900000.0 condo 3.0 2.0 97.0 0.0
6,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C477480BB48PH,3BR 97 sqm Pre Selling in Quezon City Calinea ...,11900000.0,condo,3.0,2.0,97.00,0.00,CALINEA TOWER PRE SELLING 3 BEDROOM 97 SQM IN ...,Ailyn Vigilla,https://www.lamudi.com.ph/ailyn-vigilla-agn-15...,FULLY VERIFIED,Caloocan,120.990543,14.651401,11900000.0 condo 3.0 2.0 97.0 0.0
7,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C4A17042038PH,3BR 97sqm Pre Selling in Caloocan City near No...,11900000.0,condo,3.0,2.0,97.00,0.00,CALINEA TOWER PRE SELLING 3 BEDROOM 97 SQM IN ...,Ailyn Vigilla,https://www.lamudi.com.ph/ailyn-vigilla-agn-15...,FULLY VERIFIED,Caloocan,120.990543,14.651401,11900000.0 condo 3.0 2.0 97.0 0.0
8,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C6C949EE24EPH,Pre Selling 3BR 97sqm in Caloocan City near SM...,11900000.0,condo,3.0,2.0,97.00,0.00,CALINEA TOWER PRE SELLING 3 BEDROOM 97 SQM IN ...,Ailyn Vigilla,https://www.lamudi.com.ph/ailyn-vigilla-agn-15...,FULLY VERIFIED,Caloocan,120.990543,14.651401,11900000.0 condo 3.0 2.0 97.0 0.0
41,"Amparo, Caloocan",https://www.lamudi.com.ph/projects/amparo-subd...,HO6501855B414E1PH,Corner House & Lot FOR SALE in Amparo Caloocan...,7800000.0,house,3.0,2.5,98.96,76.67,Project- SINGLE ATTACHED HOUSE & LOT Location-...,Keziah Samaniego,https://www.lamudi.com.ph/evelyn-samaniego-47598/,FULLY VERIFIED,Caloocan,121.076954,14.747185,7800000.0 house 3.0 2.5 98.96 76.67


In [47]:
title_desc_duplicates = (df["Description"]!="") & (df["Description"].duplicated()) & (df["Title"].duplicated()) & concat_duplicates
print(len(df[title_desc_duplicates]))
check = df[title_desc_duplicates]
df[title_desc_duplicates].sort_values("Title").head()

7764


,Location,URL,SKU,Title,Price,Category,Num_Bedrooms,Num_Bathrooms,Floor_Area,Land_Area,Description,Agent_Name,Agent_Links,Agent_Verifications,File,Latitude,Longitude,Concat
68540,"Cubao, Quezon City",https://www.lamudi.com.ph/buy/metro-manila/que...,HO66BEE1B39467EPH,"""Prime Investment Opportunity: 600 sqm Lot in ...",60000000.0,house,6.0,6.0,750.0,600.0,"""Discover endless possibilities with this prim...",Evelyn Perez-Dumdum,https://www.lamudi.com.ph/evelyn-perez-dumdum/,FULLY VERIFIED,Quezon City,121.04505,14.625980,60000000.0 house 6.0 6.0 750.0 600.0
68539,"Cubao, Quezon City",https://www.lamudi.com.ph/buy/metro-manila/que...,HO66BEE16252143PH,"""Prime Investment Opportunity: 600 sqm Lot in ...",60000000.0,house,6.0,6.0,750.0,600.0,"""Discover endless possibilities with this prim...",Evelyn Perez-Dumdum,https://www.lamudi.com.ph/evelyn-perez-dumdum/,FULLY VERIFIED,Quezon City,121.05720,14.617770,60000000.0 house 6.0 6.0 750.0 600.0
68561,"Pinagkaisahan, Quezon City",https://www.lamudi.com.ph/buy/metro-manila/que...,HO66BEE40DB37BBPH,"""Prime Investment Opportunity: 600 sqm Lot in ...",60000000.0,house,6.0,6.0,750.0,600.0,"""Discover endless possibilities with this prim...",Evelyn Perez-Dumdum,https://www.lamudi.com.ph/evelyn-perez-dumdum/,FULLY VERIFIED,Quezon City,121.04505,14.625980,60000000.0 house 6.0 6.0 750.0 600.0
68570,"Pinagkaisahan, Quezon City",https://www.lamudi.com.ph/buy/metro-manila/que...,HO66BEE434CBB1FPH,"""Prime Investment Opportunity: 600 sqm Lot in ...",60000000.0,house,6.0,6.0,750.0,600.0,"""Discover endless possibilities with this prim...",Evelyn Perez-Dumdum,https://www.lamudi.com.ph/evelyn-perez-dumdum/,FULLY VERIFIED,Quezon City,121.04426,14.626617,60000000.0 house 6.0 6.0 750.0 600.0
68567,"Pinagkaisahan, Quezon City",https://www.lamudi.com.ph/buy/metro-manila/que...,HO66BEE44347031PH,"""Prime Investment Opportunity: 600 sqm Lot in ...",60000000.0,house,6.0,6.0,750.0,600.0,"""Discover endless possibilities with this prim...",Evelyn Perez-Dumdum,https://www.lamudi.com.ph/evelyn-perez-dumdum/,FULLY VERIFIED,Quezon City,121.04426,14.626617,60000000.0 house 6.0 6.0 750.0 600.0


In [48]:
print(len(df))
df = df[~title_desc_duplicates]
print(len(df))

86736
78972


In [49]:
# df.to_csv("20241118 Processed File v2 - No SKU and Other Duplicates.csv", encoding="utf-8-sig", index=False)

In [50]:
#ISSUE: Seems to be same proeprty but price or other features change
df[df["Title"]=="Brand New Semi Furnished House and Lot with Pool in Greenwoods Executive Village"]

,Location,URL,SKU,Title,Price,Category,Num_Bedrooms,Num_Bathrooms,Floor_Area,Land_Area,Description,Agent_Name,Agent_Links,Agent_Verifications,File,Latitude,Longitude,Concat
47449,"San Miguel, Pasig",https://www.lamudi.com.ph/buy/metro-manila/pas...,HO65F85AC0A987FPH,Brand New Semi Furnished House and Lot with Po...,25000000.0,house,5.0,5.0,150.0,350.0,Brand New Semi Furnished House and Lot with Po...,Joshua Tanato,https://www.lamudi.com.ph/-agn-52081/,FULLY VERIFIED,Pasig,121.092265,14.567197,25000000.0 house 5.0 5.0 150.0 350.0
47465,"San Miguel, Pasig",https://www.lamudi.com.ph/buy/metro-manila/pas...,HO65F85B220034BPH,Brand New Semi Furnished House and Lot with Po...,26500000.0,house,5.0,5.0,150.0,350.0,Brand New Semi Furnished House and Lot with Po...,Joshua Tanato,https://www.lamudi.com.ph/-agn-52081/,FULLY VERIFIED,Pasig,121.092265,14.567197,26500000.0 house 5.0 5.0 150.0 350.0
48074,"San Miguel, Pasig",https://www.lamudi.com.ph/buy/metro-manila/pas...,HO66912A1781F28PH,Brand New Semi Furnished House and Lot with Po...,26500000.0,house,5.0,5.0,250.0,150.0,Brand New Semi Furnished House and Lot with Po...,Joshua Tanato,https://www.lamudi.com.ph/-agn-52081/,FULLY VERIFIED,Pasig,121.092265,14.567197,26500000.0 house 5.0 5.0 250.0 150.0
48878,"San Miguel, Pasig",https://www.lamudi.com.ph/new-developments/gre...,HO6633C2268726BPH,Brand New Semi Furnished House and Lot with Po...,25000000.0,house,5.0,5.0,350.0,150.0,Brand New Semi Furnished House and Lot with Po...,Joshua Tanato,https://www.lamudi.com.ph/-agn-52081/,FULLY VERIFIED,Pasig,121.092265,14.567197,25000000.0 house 5.0 5.0 350.0 150.0
48881,"San Miguel, Pasig",https://www.lamudi.com.ph/new-developments/gre...,HO662902C9527E1PH,Brand New Semi Furnished House and Lot with Po...,26500000.0,house,5.0,5.0,350.0,150.0,Brand New Semi Furnished House and Lot with Po...,Joshua Tanato,https://www.lamudi.com.ph/-agn-52081/,FULLY VERIFIED,Pasig,121.092265,14.567197,26500000.0 house 5.0 5.0 350.0 150.0
49140,"San Miguel, Pasig",https://www.lamudi.com.ph/buy/metro-manila/pas...,HO66795A5A17FCDPH,Brand New Semi Furnished House and Lot with Po...,25000000.0,house,5.0,5.0,350.0,50.0,Brand New Semi Furnished House and Lot with Po...,Joshua Tanato,https://www.lamudi.com.ph/-agn-52081/,FULLY VERIFIED,Pasig,121.092265,14.567197,25000000.0 house 5.0 5.0 350.0 50.0
49347,"San Miguel, Pasig",https://www.lamudi.com.ph/buy/metro-manila/pas...,HO66741D15963F1PH,Brand New Semi Furnished House and Lot with Po...,25000000.0,house,6.0,6.0,350.0,150.0,Brand New Semi Furnished House and Lot with Po...,Joshua Tanato,https://www.lamudi.com.ph/-agn-52081/,FULLY VERIFIED,Pasig,121.092265,14.567197,25000000.0 house 6.0 6.0 350.0 150.0
49394,"San Miguel, Pasig",https://www.lamudi.com.ph/buy/metro-manila/pas...,HO661950E671780PH,Brand New Semi Furnished House and Lot with Po...,2650000.0,house,5.0,5.0,350.0,150.0,Brand New Semi Furnished House and Lot with Po...,Joshua Tanato,https://www.lamudi.com.ph/-agn-52081/,FULLY VERIFIED,Pasig,121.092265,14.567197,2650000.0 house 5.0 5.0 350.0 150.0
49575,"San Miguel, Pasig",https://www.lamudi.com.ph/buy/metro-manila/pas...,HO662BD02BEE6EDPH,Brand New Semi Furnished House and Lot with Po...,25500000.0,house,5.0,5.0,350.0,150.0,Brand New Semi Furnished House and Lot with Po...,Joshua Tanato,https://www.lamudi.com.ph/-agn-52081/,FULLY VERIFIED,Pasig,121.092265,14.567197,25500000.0 house 5.0 5.0 350.0 150.0


# QGIS

In [51]:
RR = pd.read_csv("20241911v RR.csv", encoding="utf-8-sig")
RR['Matched_Zonal_Value'] = RR['Matched_Zonal_Value'].str.strip()
RR = RR[~RR['Matched_Zonal_Value'].str.contains(r'\*', na=False)]
RR['Matched_Zonal_Value'] = RR['Matched_Zonal_Value'].astype(float)
RR = RR[~RR["Matched_Zonal_Value"].isna()]
print(len(RR))

C:\Users\Leibniz\AppData\Local\Temp\ipykernel_29588\1495661556.py:1: DtypeWarning: Columns (34) have mixed types. Specify dtype option on import or set low_memory=False.
  RR = pd.read_csv("20241911v RR.csv", encoding="utf-8-sig")


45770


In [52]:
CR = pd.read_csv("20241911v CR.csv", encoding="utf-8-sig")
CR['Matched_Zonal_Value'] = CR['Matched_Zonal_Value'].str.strip()
CR = CR[~CR['Matched_Zonal_Value'].str.contains(r'\*', na=False)]
CR['Matched_Zonal_Value'] = CR['Matched_Zonal_Value'].astype(float)
CR = CR[~CR["Matched_Zonal_Value"].isna()]
print(len(CR))

3262


In [53]:
general_df = pd.merge(RR, CR, "outer")
general_df
len(general_df) == len(RR) + len(CR)

True

In [54]:
general_df.head()

,Location,URL,SKU,Title,Price,Category,Num_Bedrooms,Num_Bathrooms,Floor_Area,Land_Area,...,SchoolHubName,SchoolHubDist,BusCount,HospitalCount,MallCount,SchoolCount,Concat,Property Type,Processed_Location,Matched_Zonal_Value
0,"Amparo, Caloocan",https://www.lamudi.com.ph/projects/amparo-resi...,HO5CEE34F0060ECPH,7.2M single attached house and lot for sale at...,7200000.0,house,3.0,2.0,147.0,103.0,...,w873447606,0.342505,0,0,0,5,7200000.0 house 3.0 2.0 147.0 103.0,RR,amparo caloocan,17000.0
1,Caloocan,https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66D1B50AECE7EPH,Caloocan City RFO Condo beside Lrt Monumento,3100000.0,condo,1.0,1.0,21.7,0.0,...,w40941344,0.295216,14,7,3,15,3100000.0 condo 1.0 1.0 21.7 0.0,RR,caloocan,17000.0
2,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C1D00AA6DD1PH,Pre Selling 2BR 97sqm in Caloocan City near Ar...,11900000.0,condo,3.0,2.0,97.0,0.0,...,w40941344,0.414120,15,10,2,15,11900000.0 condo 3.0 2.0 97.0 0.0,RR,grace park east caloocan,23000.0
3,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C6B155BF32BPH,3BR 97sqm Pre Selling in Caloocan City near Sq...,11900000.0,condo,3.0,2.0,97.0,0.0,...,w40941344,0.414120,15,10,2,15,11900000.0 condo 3.0 2.0 97.0 0.0,RR,grace park east caloocan,23000.0
4,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C1EFCC46FD2PH,2 Bedroom Preselling units in The Calinea Towe...,8615000.0,condo,2.0,1.0,57.5,0.0,...,w40941344,0.414120,15,10,2,15,8615000.0 condo 2.0 1.0 57.5 0.0,RR,grace park east caloocan,23000.0


In [ ]:
# general_df.to_csv("general_df.csv", encoding="utf-8-sig", index=False)

## Second Cleaning

In [56]:
qgis_df = pd.read_csv("20241119v Properties with Distance and Count.csv", encoding="utf-8-sig")
print(len(qgis_df))
qgis_df.head()

80223


,Location,URL,SKU,Title,Price,Category,Num_Bedroo,Num_Bathro,Floor_Area,Land_Area,...,HospitalHubName,HospitalHubDist,MallHubName,MallHubDist,SchoolHubName,SchoolHubDist,BusCount,HospitalCount,MallCount,SchoolCount
0,"Amparo, Caloocan",https://www.lamudi.com.ph/projects/amparo-resi...,HO5CEE34F0060ECPH,7.2M single attached house and lot for sale at...,7200000.0,house,3.0,2.0,147.0,103.0,...,n255061867,1.316144,SM City Fairview,2.509902,w873447606,0.342505,0,0,0,5
1,Caloocan,https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66D1B50AECE7EPH,Caloocan City RFO Condo beside Lrt Monumento,3100000.0,condo,1.0,1.0,21.7,0.0,...,n255050207,0.138596,North Mall,0.708891,w40941344,0.295216,14,7,3,15
2,"Amparo, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,LA66C596CFA8463PH,"850sqm Vacant Lot in Makabud Street, Amparo, N...",17000000.0,land,0.0,0.0,0.0,850.0,...,n255061867,1.316144,SM City Fairview,2.509902,w873447606,0.342505,0,0,0,5
3,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C1D00AA6DD1PH,Pre Selling 2BR 97sqm in Caloocan City near Ar...,11900000.0,condo,3.0,2.0,97.0,0.0,...,n255066691,0.261535,North Mall,0.698302,w40941344,0.414120,15,10,2,15
4,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C6B155BF32BPH,3BR 97sqm Pre Selling in Caloocan City near Sq...,11900000.0,condo,3.0,2.0,97.0,0.0,...,n255066691,0.261535,North Mall,0.698302,w40941344,0.414120,15,10,2,15


In [57]:
qgis_df.rename(columns={"Num_Bedroo": "Num_Bedrooms", "Num_Bathro": "Num_Bathrooms", "Descriptio": "Description", "Agent_Veri": "Agent_Verification"}, inplace=True)

In [58]:
# Fixing data type
for var in ["Price", "Num_Bedrooms", "Num_Bathrooms", "Floor_Area", "Land_Area"]:
    qgis_df[var].fillna(0, inplace=True)
    qgis_df[var] = pd.to_numeric(qgis_df[var], errors='coerce')
for var in ["Title", "Description"]:
    qgis_df[var].fillna("", inplace=True)
    qgis_df[var] = qgis_df[var].astype(str)

# Translating common accented characters
qgis_df.replace(r"Ã±", "ñ", regex=True, inplace=True)
qgis_df.replace(r"Ã‘", "Ñ", regex=True, inplace=True)
qgis_df.replace(r"Ã©", "é", regex=True, inplace=True)

# Removing all characters that are not letters, numbers, common punctuation, or ñÑé
# Reduce any amount of whitespace to a single space
def clean_text(text):
    cleaned_text = re.sub(r"[^a-zA-Z0-9.,!? \'\"ñÑé]+", " ", text)
    cleaned_text = re.sub(r"\s+", " ", cleaned_text)
    
    return cleaned_text

qgis_df['Title'] = qgis_df['Title'].apply(clean_text)
qgis_df['Description'] = qgis_df['Description'].apply(clean_text)

In [59]:
rent_properties = qgis_df[qgis_df["URL"].str.contains(r".*rent.*")]
rent_properties.head()

,Location,URL,SKU,Title,Price,Category,Num_Bedrooms,Num_Bathrooms,Floor_Area,Land_Area,...,HospitalHubName,HospitalHubDist,MallHubName,MallHubDist,SchoolHubName,SchoolHubDist,BusCount,HospitalCount,MallCount,SchoolCount
23,"Amparo, Caloocan",https://www.lamudi.com.ph/projects/amparo/rent...,HO63A31ECE3C309PH,Zero Down Rent to Own Townhouse 3BR. Move in R...,2500000.0,house,3.0,1.0,40.00,19.00,...,n255061867,1.316144,SM City Fairview,2.509902,w873447606,0.342505,0,0,0,5
87,"Amparo, Caloocan",https://www.lamudi.com.ph/projects/amparo-subd...,HO5CDA7F1A7951FPH,"RFO, Rent to Own, Townhouse For Sale at Mangga...",5800000.0,house,3.0,2.0,123.52,76.91,...,r14713258,1.359740,SM City Fairview,4.033290,w932362768,0.480251,5,0,0,7
123,"Amparo, Caloocan",https://www.lamudi.com.ph/projects/amparo-subd...,HO5CEE3AEEA9BBAPH,10 DP Rent To Own Townhouse For Sale at Amparo...,5800000.0,house,3.0,2.0,123.00,76.00,...,r14713258,1.359740,SM City Fairview,4.033290,w932362768,0.480251,5,0,0,7
124,"Amparo, Caloocan",https://www.lamudi.com.ph/projects/amparo-subd...,HO5CBAC96FCE0BEPH,"Unit 7, 10 Down only, Rent to own Townhouse Fo...",5520000.0,house,3.0,2.0,123.00,76.00,...,r14713258,1.359740,SM City Fairview,4.033290,w932362768,0.480251,5,0,0,7
186,"Amparo, Caloocan",https://www.lamudi.com.ph/projects/amparo-subd...,HO5CEE32D336289PH,"RFO, Rent To Own townhouse For Sale at Amparo ...",5800000.0,house,3.0,2.0,123.00,76.00,...,n255061867,1.316144,SM City Fairview,2.509902,w873447606,0.342505,0,0,0,5


In [60]:
qgis_df["Concat"] = qgis_df.iloc[:, 4:10].apply(lambda row: ' '.join(row.values.astype(str)), axis=1)
concat_duplicates = qgis_df["Concat"].duplicated()
qgis_df[concat_duplicates].head()

,Location,URL,SKU,Title,Price,Category,Num_Bedrooms,Num_Bathrooms,Floor_Area,Land_Area,...,HospitalHubDist,MallHubName,MallHubDist,SchoolHubName,SchoolHubDist,BusCount,HospitalCount,MallCount,SchoolCount,Concat
4,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C6B155BF32BPH,3BR 97sqm Pre Selling in Caloocan City near Sq...,11900000.0,condo,3.0,2.0,97.00,0.00,...,0.261535,North Mall,0.698302,w40941344,0.414120,15,10,2,15,11900000.0 condo 3.0 2.0 97.0 0.0
6,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C477480BB48PH,3BR 97 sqm Pre Selling in Quezon City Calinea ...,11900000.0,condo,3.0,2.0,97.00,0.00,...,0.261535,North Mall,0.698302,w40941344,0.414120,15,10,2,15,11900000.0 condo 3.0 2.0 97.0 0.0
7,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C4A17042038PH,3BR 97sqm Pre Selling in Caloocan City near No...,11900000.0,condo,3.0,2.0,97.00,0.00,...,0.261535,North Mall,0.698302,w40941344,0.414120,15,10,2,15,11900000.0 condo 3.0 2.0 97.0 0.0
8,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C6C949EE24EPH,Pre Selling 3BR 97sqm in Caloocan City near SM...,11900000.0,condo,3.0,2.0,97.00,0.00,...,0.261535,North Mall,0.698302,w40941344,0.414120,15,10,2,15,11900000.0 condo 3.0 2.0 97.0 0.0
33,"Amparo, Caloocan",https://www.lamudi.com.ph/projects/amparo-subd...,HO6501855B414E1PH,Corner House Lot FOR SALE in Amparo Caloocan C...,7800000.0,house,3.0,2.5,98.96,76.67,...,1.316144,SM City Fairview,2.509902,w873447606,0.342505,0,0,0,5,7800000.0 house 3.0 2.5 98.96 76.67


In [61]:
title_desc_duplicates = (qgis_df["Description"]!="") & (qgis_df["Description"].duplicated()) & (qgis_df["Title"].duplicated()) & concat_duplicates
print(len(qgis_df[title_desc_duplicates]))
check = qgis_df[title_desc_duplicates]
qgis_df[title_desc_duplicates].sort_values("Title").head()

8204


,Location,URL,SKU,Title,Price,Category,Num_Bedrooms,Num_Bathrooms,Floor_Area,Land_Area,...,HospitalHubDist,MallHubName,MallHubDist,SchoolHubName,SchoolHubDist,BusCount,HospitalCount,MallCount,SchoolCount,Concat
55024,"Culiat, Quezon City",https://www.lamudi.com.ph/buy/metro-manila/que...,HO6655891E43951PH,2 Q,10600000.0,house,3.0,3.0,150.0,92.0,...,0.994810,UP Shopping Center,2.134954,n1226627801,0.267532,2,1,0,11,10600000.0 house 3.0 3.0 150.0 92.0
60555,"Holy Spirit, Quezon City",https://www.lamudi.com.ph/buy/metro-manila/que...,HO6694CA629AEF2PH,21.8 3 2 Q,21800000.0,house,4.0,4.0,259.0,100.0,...,1.362485,Ever Gotesco Commonwealth,0.941136,w800211631,0.371361,0,0,1,15,21800000.0 house 4.0 4.0 259.0 100.0
60537,"Holy Spirit, Quezon City",https://www.lamudi.com.ph/buy/metro-manila/que...,HO6694CA1F1C874PH,21.8 3 2 Q,21800000.0,house,4.0,4.0,259.0,100.0,...,1.362485,Ever Gotesco Commonwealth,0.941136,w800211631,0.371361,0,0,1,15,21800000.0 house 4.0 4.0 259.0 100.0
60521,"Holy Spirit, Quezon City",https://www.lamudi.com.ph/buy/metro-manila/que...,HO6694CA46703C6PH,21.8 3 2 Q,21800000.0,house,4.0,4.0,259.0,100.0,...,1.362485,Ever Gotesco Commonwealth,0.941136,w800211631,0.371361,0,0,1,15,21800000.0 house 4.0 4.0 259.0 100.0
59091,"Kamuning, Quezon City",https://www.lamudi.com.ph/buy/metro-manila/que...,HO66962C122ABB7PH,24.5 3 Q,24500000.0,house,3.0,4.0,236.0,80.0,...,0.406967,Robinson’s Magnolia,1.374013,w135723970,0.123822,43,3,0,19,24500000.0 house 3.0 4.0 236.0 80.0


In [62]:
print(len(qgis_df))
qgis_df = qgis_df[~title_desc_duplicates]
print(len(qgis_df))

80223
72019


In [ ]:
# qgis_df.to_csv("QGIS_df.csv", encoding="utf-8-sig", index=False)

# Merge w/ other Independent Data

In [12]:
import pandas as pd
import os
qgis_df = pd.read_csv("QGIS_df.csv", encoding="utf-8-sig")
qgis_df.head()

,Location,URL,SKU,Title,Price,Category,Num_Bedrooms,Num_Bathrooms,Floor_Area,Land_Area,...,HospitalHubDist,MallHubName,MallHubDist,SchoolHubName,SchoolHubDist,BusCount,HospitalCount,MallCount,SchoolCount,Concat
0,"Amparo, Caloocan",https://www.lamudi.com.ph/projects/amparo-resi...,HO5CEE34F0060ECPH,7.2M single attached house and lot for sale at...,7200000.0,house,3.0,2.0,147.0,103.0,...,1.316144,SM City Fairview,2.509902,w873447606,0.342505,0,0,0,5,7200000.0 house 3.0 2.0 147.0 103.0
1,Caloocan,https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66D1B50AECE7EPH,Caloocan City RFO Condo beside Lrt Monumento,3100000.0,condo,1.0,1.0,21.7,0.0,...,0.138596,North Mall,0.708891,w40941344,0.295216,14,7,3,15,3100000.0 condo 1.0 1.0 21.7 0.0
2,"Amparo, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,LA66C596CFA8463PH,"850sqm Vacant Lot in Makabud Street, Amparo, N...",17000000.0,land,0.0,0.0,0.0,850.0,...,1.316144,SM City Fairview,2.509902,w873447606,0.342505,0,0,0,5,17000000.0 land 0.0 0.0 0.0 850.0
3,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C1D00AA6DD1PH,Pre Selling 2BR 97sqm in Caloocan City near Ar...,11900000.0,condo,3.0,2.0,97.0,0.0,...,0.261535,North Mall,0.698302,w40941344,0.414120,15,10,2,15,11900000.0 condo 3.0 2.0 97.0 0.0
4,"Grace Park East, Caloocan",https://www.lamudi.com.ph/buy/metro-manila/cal...,CD66C6B155BF32BPH,3BR 97sqm Pre Selling in Caloocan City near Sq...,11900000.0,condo,3.0,2.0,97.0,0.0,...,0.261535,North Mall,0.698302,w40941344,0.414120,15,10,2,15,11900000.0 condo 3.0 2.0 97.0 0.0


In [22]:
cmci_data = pd.read_csv(os.path.join("Independent Data", "CMCI_2023_Shortlist.csv"))
cmci_data.head()

,PROVINCE / LGU,employment_generation,financial_deepening,local_economy_growth,local_economy_size,presence_of_business_and_professional_organizations,safety_compliant_business
0,Quezon (MM),0.9248,1.3115,0.5111,0.4807,0.1880,1.2403
1,Caloocan,0.1654,0.4734,0.3167,0.0555,0.0567,0.8881
2,Manila,0.4600,1.1539,0.3905,0.3591,0.3590,0.6528
3,Pasay,2.0000,0.5909,0.2022,2.0000,0.1095,1.2830
4,Paranaque,0.7071,0.3316,0.2952,0.3623,0.0081,0.3482
